# 조업이벤트·정기수리 기록이 온도 데이터 기간과 겹치는지 확인하기

**담당: 박민호**  ·  관련 Issue: #10 (번호를 채우세요)

## 이 노트북에서 할 일

조업이벤트(G-02)와 정기수리(G-01)의 날짜 범위를 확인하고,
온도 데이터 기간과 **실제로 겹치는지**, 겹친다면 **몇 건이나 되는지** 세어 봅니다.

## 왜 하는지

온도만 보면 "이상하다"로 끝나지만, 그 시각에 조업 조건이 바뀌었거나 정기수리 중이었다면
**이상이 아니라 예정된 작업**입니다. 이 대조가 되어야 이상 탐지 결과를 설명할 수 있습니다.

그런데 세 파일의 기간이 서로 안 겹치면 대조 자체가 불가능합니다. 그래서 가장 먼저 확인해야 합니다.

## 진행 방법

1. 아래 칸을 **위에서부터 순서대로** 실행하세요.
2. `# TODO` 라고 적힌 곳을 직접 채우세요. 막히면 디스코드에 물어보세요.
3. 맨 아래 **결과 정리** 칸에 확인한 값을 한국어 문장으로 적으세요. 이게 진짜 결과물입니다.
4. 다 했으면 커밋 전에 맨 마지막 안내를 읽으세요.

> 데이터 파일이 없으면 `data/` 폴더에 CSV 3개를 먼저 넣으세요. 이 파일들은 Git에 올라가지 않습니다(공유받은 자료를 각자 직접 넣습니다).

## 공통 준비

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로를 잡아둡니다.
import os
import pandas as pd

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
TEMP_CSV   = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV  = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")

pd.set_option("display.max_columns", 50)
print("경로 확인:", os.path.exists(TEMP_CSV), os.path.exists(EVENT_CSV), os.path.exists(REPAIR_CSV))

### 1단계 — 세 파일을 모두 읽으세요

In [ ]:
df_temp   = pd.read_csv(TEMP_CSV, encoding="utf-8")
df_event  = pd.read_csv(EVENT_CSV, encoding="utf-8")
df_repair = pd.read_csv(REPAIR_CSV, encoding="utf-8")

for name, d in [("온도", df_temp), ("조업이벤트", df_event), ("정기수리", df_repair)]:
    print(name, d.shape)
    print(list(d.columns))
    print()

### 2단계 — 각 파일의 시각 컬럼을 날짜 형식으로 바꾸세요

파일마다 시각 컬럼 이름이 다릅니다.

| 파일 | 시각 컬럼 |
| --- | --- |
| 온도 | `MEAS_DT` |
| 조업이벤트 | `EVT_DT` |
| 정기수리 | `STRT_DT`(시작), `END_DT`(끝) |

In [ ]:
# TODO: 각 데이터프레임의 시각 컬럼을 pd.to_datetime 으로 바꾸세요.
df_temp['MEAS_DT'] = pd.to_datetime(df_temp['MEAS_DT'])
display(df_temp[["MEAS_DT"]])
print()
df_event['EVT_DT'] = pd.to_datetime(df_event['EVT_DT'])
display([df_event["EVT_DT"]])
print()
df_repair['STRT_DT'] = pd.to_datetime(df_repair['STRT_DT'])
df_repair['END_DT'] = pd.to_datetime(df_repair['END_DT'])
display(df_repair[["STRT_DT","END_DT"]])

### 3단계 — 각 파일의 기간(시작~끝)을 출력하세요

In [ ]:
# TODO: 세 파일의 최소/최대 시각을 각각 출력하세요.
# 힌트: df_temp["MEAS_DT"].min(), df_temp["MEAS_DT"].max() ...

temp_strt, temp_end = df_temp["MEAS_DT"].min(), df_temp["MEAS_DT"].max()
print(f"온도 파일 시작일: {temp_strt} | 종료일: {temp_end}")
event_strt, event_end = df_event["EVT_DT"].min(), df_event["EVT_DT"].max()
print(f"조업이벤트.csv 시작일: {event_strt} | 종료일: {event_end}")
repair_strt, repair_end = df_repair["STRT_DT"].min(), df_repair["STRT_DT"].max()
print(f"정기수리캘린더.csv 시작일: {repair_strt} | 종료일: {repair_end}")

### 4단계 — 온도 데이터 기간 안에 들어오는 건수를 세세요

온도 데이터의 시작~끝 사이에 들어오는 이벤트가 몇 건인지 세어 봅니다.
0건이면 대조가 불가능하다는 뜻이니, 그 사실을 꼭 기록해야 합니다.

In [ ]:
# TODO: 조업이벤트 중 temp_strt ~ temp_end 사이에 들어오는 건수를 세세요.
# 힌트: mask = (df_event["EVT_DT"] >= temp_strt) & (df_event["EVT_DT"] <= temp_end)
#       print(mask.sum(), "건")
mask_event = (df_event["EVT_DT"] >= temp_strt) & (df_event["EVT_DT"] <= temp_end)
print("조업이벤트 날짜 사이에 들어오는 건수:", mask_event.sum(), "건")

# TODO: 정기수리는 구간(STRT_DT ~ END_DT)이라 겹침 판정이 조금 다릅니다.
# 두 구간이 겹치는 조건: 수리 시작 <= 온도 끝  그리고  수리 끝 >= 온도 시작
mask_repair = (df_repair["STRT_DT"] <= temp_end) & (df_repair["END_DT"] >= temp_strt)
print("정기 수리 기간과 겹치는 건수:", mask_repair.sum(), "건")

### 5단계 — 설비 코드도 맞는지 확인하세요

기간이 겹쳐도 **다른 설비의 기록**이면 소용이 없습니다.
조업이벤트와 정기수리에는 `PLANT_CD`(설비 코드)가 있습니다.
우리 온도 데이터는 `T-CR1-CAL01`, 즉 CR1 라인입니다.

In [ ]:
# TODO: 두 파일의 PLANT_CD 값 종류를 확인하고, CR1 관련 코드가 있는지 보세요.
# 힌트: df_event["PLANT_CD"].value_counts()
print("조업 이벤트 CR1", df_event["PLANT_CD"].value_counts())
print()
print("정기수리캘린더 CR1", df_repair["PLANT_CD"].value_counts())

## 결과 정리 — 여기를 꼭 채우세요

아래 빈칸을 확인한 값으로 바꿔서 적으세요. 이 내용이 `docs/02-data-contract.md`로 옮겨집니다.

| 파일 | 시작 | 끝 | 온도 기간과 겹치는 건수 |
| --- | --- | --- | --- |
| 온도 (T-CR1-CAL01) | 2024-01-02 03:36:00 | 2024-12-28 16:44:24 | — |
| 조업이벤트 (G-02) | 2024-01-02 03:36:00 | 2024-12-31 20:15:00 | 262 건 |
| 정기수리 (G-01) | 2024-03-15 00:00:00 | 2024-11-05 00:00:00 | 4 건 |

**결론:** 세 데이터를 시간으로 대조할 수 있다 / 없다 → 있다


### 이상하다고 느낀 점 / 확실하지 않은 점

- (있으면 적어주세요. 없으면 "없음"이라고 적으세요.) 없음

---

## 커밋하기 전에 읽으세요

1. **출력은 지우지 않아도 됩니다.** `nbstripout` 필터를 등록해 뒀다면 `git add` 할 때 자동으로 지워집니다.
   등록했는지 확인: `python -m nbstripout --status` → `Automatic cleanup enabled` 가 나와야 합니다.
   안 나오면: `python -m nbstripout --install --attributes .gitattributes`
2. 이 노트북 **한 파일만** 커밋하세요. 다른 사람 파일은 건드리지 마세요.
3. 브랜치를 만들어서 작업하세요. 예) `git switch -c feat/이슈번호-설명`
4. 커밋 메시지 첫 줄: `feat: 세 데이터의 기간 겹침 확인`
5. PR 본문에 `Closes #이슈번호` 를 꼭 적으세요.